# ScreamingFace · DRACO Preview

Run one real DRACO research question through the complete ScreamingFace workflow: connect,
compose, evaluate, and compare.

This notebook uses `draco-preview@1`, not canonical `draco@1`. Preview keeps the pinned real
100-question source, one real positive rubric criterion per question, the official per-criterion
judge prompt, and the same grading and aggregation implementation. One case makes six model calls:
two research calls, one synthesis call, and three judge calls.

It deliberately differs from canonical DRACO:

| | DRACO Preview | Canonical DRACO |
|---|---|---|
| research | two independently prompted Claude calls | seven-model, nine-Fusion lineup |
| search | engine-hosted SearXNG | earlier reproduction used OpenRouter |
| synthesis | Codex GPT-5.5 | pinned pipeline-specific synthesizers |
| judging | Gemini 2.5 Flash, one criterion, one pass | Gemini 3.1 Pro, full rubric, three passes |
| claim | architecture validation | score-comparable reproduction |

The two Claude calls are independent: one builds an evidence-led answer and the other searches for
omissions and counterevidence. Both receive `web_search`. Codex is the model reducer because its
current route is tool-free; it combines the resolved research answers without performing another
search. Gemini 2.5 Flash is tool-free and used only as the judge. Gemini 3 research is not
advertised because its function-calling continuation requires an encrypted `thoughtSignature`
that the current AI Gateway normalization does not preserve.

The earlier reproduction routed research and its Gemini 3.1 Pro judge through OpenRouter. AI
Gateway OpenRouter support is tracked in OME-428. Until that lands, Preview must not be presented
as a DRACO score or compared with the paper. A typical canonical case would make about 354 judge
calls for this three-answer target shape; Preview makes three.

## Before you run it

Start the local stack:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

It provides the ScreamingFace engine at `http://127.0.0.1:4404`, AI Gateway behind it, and private
SearXNG search. Only the engine contacts AI Gateway. Each case is sent as one
`GET /v1?q=<URL-encoded-expression>` URL4 request; the engine returns plaintext that the SDK parses
into immutable values.

DRACO is downloaded by this Python process through your Hugging Face session. Run
`huggingface-cli login` if the dataset requires it; that token is never sent to the engine. Connect
Claude, Codex, and Gemini below before evaluating. The live progress panel reports actual cases and
judge responses while the synchronous call runs.

## 1 · Connect

In [ ]:
import screamingface as sf

sf.connect()

## 2 · Compose

In [ ]:
EVIDENCE_PROMPT = """Research the question before answering. Build an evidence-led
account with specific facts and sources, cover every part of the prompt, and return
clear, structured prose."""

CHALLENGE_PROMPT = """Research the question independently. Look for omissions,
counterevidence, and weak assumptions before producing a complete answer with
specific facts and sources."""

SYNTHESIS_PROMPT = """Produce one comprehensive answer by combining the strongest
facts, arguments, and citations from every labeled member answer. Resolve
disagreements in favor of the more specific and better-supported claim. Return only
the unified prose answer."""

fusion = sf.Fusion(
    "draco-research-duo",
    models=[
        {"model": "claude/sonnet-4.6", "prompt": EVIDENCE_PROMPT},
        {"model": "claude/sonnet-4.6", "prompt": CHALLENGE_PROMPT},
    ],
    reducer=sf.reducers.Model(
        model="codex/gpt-5.5",
        prompt=SYNTHESIS_PROMPT,
    ),
)

fusion

## 3 · Evaluate

In [ ]:
report = fusion.evaluate("draco-preview@1", first=1)

# Equivalent staged API:
# benchmark = sf.benchmarks.load("draco-preview@1")
# run = fusion.run(benchmark, first=1)
# grades = run.grade()
# report = grades.aggregate()

## 4 · Compare

In [ ]:
report